In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


In [4]:
# defining the state
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [5]:
llm  = HuggingFaceEndpoint(
        repo_id = "meta-llama/Meta-Llama-3-8B-Instruct",
        task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

def chat_node(state: ChatState):

    # take user query from the state:
    messages = state['messages']

    # send it to LLM
    response = model.invoke(messages)

    # store response in the state
    return {'messages': [response]}

In [ ]:
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

# add edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile()